## Getting Started

### Install Vertex AI SDK for Python and other dependencies

In [ ]:
%pip install -U -q google-cloud-aiplatform langchain-core langchain-google-vertexai langchain-text-splitters langchain-community "unstructured[all-docs]" pypdf pydantic lxml pillow matplotlib opencv-python

In [ ]:
!pip install llama-parse
!pip install -qU langchain-text-splitters

### Restart current runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which will restart the current kernel.

In [ ]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

<div class="alert alert-block alert-warning">
<b>⚠️ The kernel is going to restart. Please wait until it is finished before continuing to the next step. ⚠️</b>
</div>




### Authenticate your notebook environment (Colab only)



If you are running this notebook on Google Colab, run the following cell to authenticate your environment. This step is not required if you are using [Vertex AI Workbench](https://cloud.google.com/vertex-ai-workbench).


In [ ]:
import sys

# Additional authentication is required for Google Colab
if "google.colab" in sys.modules:
    # Authenticate user to Google Cloud
    from google.colab import auth

    auth.authenticate_user()

WARNING: google.colab.auth.authenticate_user() is not supported in Colab Enterprise.


### Define Google Cloud project information

In [ ]:
PROJECT_ID = "XXXXX"  # @param {type:"string"}
LOCATION = "XXXXX"  # @param {type:"string"}

GCS_BUCKET = "XXXXX"  # @param {type:"string"}
GCS_BUCKET_URI = f"gs://{GCS_BUCKET}"

### Initialize the Vertex AI SDK

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=GCS_BUCKET_URI)

### Import libraries

In [ ]:
import base64
import io
import json
import os
import uuid
import re
import time
from google.cloud import storage
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_google_vertexai import (
    ChatVertexAI,
    VectorSearchVectorStore,
    VertexAIEmbeddings,
)
from langchain_google_vertexai.vectorstores.document_storage import GCSDocumentStorage
from langchain_text_splitters import MarkdownHeaderTextSplitter
from PIL import Image
from typing import List
import urllib.parse
from uuid import uuid4

### Define model information

- [Vertex AI - Model Information](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models)

In [ ]:
MODEL_NAME = "gemini-2.5-flash-preview-04-17"
GEMINI_OUTPUT_TOKEN_LIMIT = 8192

EMBEDDING_MODEL_NAME = "text-embedding-004"
EMBEDDING_TOKEN_LIMIT = 4096

TOKEN_LIMIT = min(GEMINI_OUTPUT_TOKEN_LIMIT, EMBEDDING_TOKEN_LIMIT)

### Define document information

In [ ]:
FOLDER_NAME = "XXXXXXXX"  # @param {type:"string"}
CHUNK_IDS_PATH = f"{GCS_BUCKET}/corpus/{FOLDER_NAME}/chunk_ids.json"
DOCUMENT_IDS_PATH = f"{GCS_BUCKET}/corpus/{FOLDER_NAME}/document_ids.json"

## Create Chunks

### Files & Images Downloading

In [ ]:
def download_all_from_bucket(bucket_name, gcs_path, local_base_path):
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blobs = list(bucket.list_blobs(prefix=gcs_path))

    for blob in blobs:
        relative_path = os.path.relpath(blob.name, gcs_path)  # שומר על כל המבנה
        local_path = os.path.join(local_base_path, relative_path)

        local_dir = os.path.dirname(local_path)
        os.makedirs(local_dir, exist_ok=True)

        if not blob.name.endswith("/"):
            blob.download_to_filename(local_path)
            print(f"✅ Downloaded {blob.name} to {local_path}")


download_all_from_bucket(
    bucket_name=GCS_BUCKET, gcs_path="corpus/", local_base_path="/content/corpus"
)

### Create documents chunks from MD files

In [ ]:
FILE_PATH = "/content/XXXXX/XXXXX.md"  # @param {type:"string"}
FILE_NAME = "XXXXX.pdf"  # @param {type:"string"}

In [ ]:
with open(FILE_PATH, "r", encoding="utf-8") as f:
    md_text = f.read()

In [ ]:
page_chunks = md_text.split("\n---\n")
documents_readyMD = [
    Document(
        page_content=chunk.strip(),
        metadata={"page_number": idx + 1, "file_name": FILE_NAME},
    )
    for idx, chunk in enumerate(page_chunks)
]

In [ ]:
def split_markdown_to_chunks(md_text, file_name) -> List[Document]:
    headers_to_split_on = [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")]
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)

    page_chunks = md_text.split("\n---\n")
    final_documents = []

    for idx, page_text in enumerate(page_chunks):
        page_number = idx + 1
        split_docs = splitter.split_text(page_text)

        for doc in split_docs:
            final_documents.append(
                Document(
                    page_content=doc.page_content,
                    metadata={
                        **doc.metadata,
                        "page_number": page_number,
                        "filename": file_name,
                    },
                )
            )

    return final_documents


def assign_chunk_ids(chunks):
    for chunk in chunks:
        chunk.metadata["doc_id"] = str(uuid4())
    return chunks


def looks_like_orphan(text):
    text = text.strip()
    if not text:
        return False
    first_line = text.splitlines()[0].strip()
    is_header = re.match(r"^#{1,3} ", first_line)
    is_starting_upper = first_line and first_line[0].isupper()
    return not is_header and not is_starting_upper


def extract_last_header_and_text(text: str) -> tuple[str, str, str] | None:
    """
    Locates the last title (including header type) and saves all text from the title itself to the end of the chunk.
    """
    lines = text.strip().splitlines()
    for i in reversed(range(len(lines))):
        line = lines[i]
        if line.startswith("### "):
            return "Header 3", line.strip(), "\n".join(lines[i:])
        elif line.startswith("## "):
            return "Header 2", line.strip(), "\n".join(lines[i:])
        elif line.startswith("# "):
            return "Header 1", line.strip(), "\n".join(lines[i:])
    return None


def extract_text_until_next_header(text) -> str:
    lines = text.strip().splitlines()
    result = []
    for line in lines:
        if re.match(r"^#{1,3} ", line):
            break
        result.append(line)
    return "\n".join(result)


def extract_complete_section_chunks(chunks):
    extended_chunks = []

    for i in range(1, len(chunks)):
        curr = chunks[i]
        prev = chunks[i - 1]

        is_orphan = not any(
            k in curr.metadata for k in ["Header 1", "Header 2", "Header 3"]
        ) and looks_like_orphan(curr.page_content)
        if not is_orphan:
            continue

        header_info = extract_last_header_and_text(prev.page_content)
        if not header_info:
            continue

        header_key, header_value, prev_body = header_info
        curr_body = extract_text_until_next_header(curr.page_content)

        if not (prev_body.strip() or curr_body.strip()):
            continue

        full_text = f"{prev_body.strip()}\n{curr_body.strip()}"
        new_chunk_id = str(uuid4())

        prev.metadata["related_node_info"] = new_chunk_id
        curr.metadata["related_node_info"] = new_chunk_id

        extended_chunks.append(
            Document(
                page_content=full_text,
                metadata={
                    header_key: header_value,
                    "doc_id": new_chunk_id,
                    "filename": prev.metadata["filename"],
                    "related_node_info": [
                        prev.metadata["doc_id"],
                        curr.metadata["doc_id"],
                    ],
                    "page_number": prev.metadata["page_number"],
                    "page_number_start": prev.metadata["page_number"],
                    "page_number_end": curr.metadata["page_number"],
                },
            )
        )
    chunks += extended_chunks
    return chunks

In [ ]:
def process_markdown_file(md_text: str, file_name: str) -> List[Document]:
    chunks = split_markdown_to_chunks(md_text, file_name)
    chunks_with_ids = assign_chunk_ids(chunks)
    chunks_with_updated_metadata = extract_complete_section_chunks(chunks_with_ids)
    return chunks_with_updated_metadata

In [ ]:
chunks_with_updated_metadata = process_markdown_file(md_text, FILE_NAME)

### Create images chunks

Use **Summarization & Saving** section to create new summaries or **Summaries loading** section for using with existing summaries.

In [ ]:
def resize_base64_image(base64_string, scale=0.5):
    img_data = base64.b64decode(base64_string)
    img = Image.open(io.BytesIO(img_data))

    new_width = int(img.width * scale)
    new_height = int(img.height * scale)

    resized_img = img.resize((new_width, new_height), Image.LANCZOS)

    buffered = io.BytesIO()
    resized_img.save(buffered, format=img.format)

    return base64.b64encode(buffered.getvalue()).decode("utf-8")

In [ ]:
def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

#### Summarization & Saving

In [ ]:
SUMMARIZATION_PROMPT = """You are an assistant tasked with summarizing visual content (e.g., diagrams, tables, graphs, or schematics) related to systems, performance, and components of the AW119 "Koala" helicopter (referred to as "Ofer"). These summaries will be embedded and used for retrieving the corresponding image based on specific questions or queries.

### General Instructions:
1. **Focus**: Identify and highlight critical terms, labels, thresholds, and unique identifiers visible in the image, emphasizing elements likely to appear in queries.
2. **File Metadata**: Always include the file name or identifier and a general description of what the image contains at the beginning of the summary. For example: "Figure 9-4: Performance graph showing...".
3. **Context and Conditions**:
   - Explicitly state the context or conditions being depicted (e.g., flight configuration, test environment, operational conditions, or system state).
   - Emphasize what is being tested or demonstrated in the image, especially if the information is similar to other visuals.
4. **Uniqueness**: Ensure summaries are tailored to the specific image by incorporating unique labels, terminology, and identifiers that distinguish it from similar visuals.

### Specific Instructions by Image Type:
1. **Diagrams or Schematics**:
   - Describe the structure, purpose, and connections between components.
   - Include key labels, visual markers, and relationships between elements.
   - Mention unique visual or structural details and the context of the system or operation depicted.

2. **Tables**:
   - Extract all data points in a structured and concise format.
   - Emphasize comparative metrics, unique values, significant thresholds, and the context in which the data applies.

3. **Graphs**:
   - Highlight the axes, trends, and critical points such as thresholds, inflection points, or labeled zones.
   - Include context for the graph, such as the operational condition, system state, or test scenario.
   - Incorporate unique elements like curve names, weights, or operational limits.

4. **Mixed Visuals (e.g., graphs with annotations, schematics with overlays)**:
   - Provide a unified summary combining elements from all relevant categories, emphasizing the primary purpose, context, and unique details of the image.

### Formatting:
- Always start the summary with the file name or figure label (if available) followed by a concise, descriptive opening. Example: "Figure 9-4: Performance graph showing..." or "File 'FuelSystem123.png': Diagram illustrating the fuel system...".
- Explicitly state the context, configuration, or test conditions visible in the image. Example: "Figure 12-3: Graph showing engine performance under standard temperature and pressure conditions (15°C, sea level)."
- Use structured phrasing that facilitates quick comprehension and retrieval. Avoid excessive details unrelated to the image's purpose.

### Important Notes:
- Do not include assumptions, interpretations, or information not explicitly visible in the image.
- Avoid introductory phrases, comments, or notes; start directly with the essential details.
- Ensure clarity and technical accuracy in every summary.

### Examples:
- **Diagram**: "File 'FuelFlow123.png': Diagram of a fuel system illustrating tanks, pumps, and valves, with labeled connections showing fuel flow directions and pressure thresholds under Cruise configuration."
- **Graph**: "Figure 9-4: Performance graph showing true airspeed versus torque for Cruise configuration with gross weight curves labeled for 2050 kg to 2850 kg and thresholds for MCP and TOP."
- **Table**: "File 'EngineSpecs456.png': Table comparing engine parameters, including power output (kW), fuel consumption (kg/h), and operating limits across configurations for Takeoff and Cruise conditions."

### Objective:
Ensure each summary captures the unique aspects of the image, includes its file name or label, and explicitly describes the context or conditions depicted to enable precise retrieval during question-answering tasks.
    """

In [ ]:
def image_summarize(base64_image: str) -> str:
    model = ChatVertexAI(model_name=MODEL_NAME, max_output_tokens=TOKEN_LIMIT)
    msg = model.invoke(
        [
            HumanMessage(
                content=[
                    {"type": "text", "text": SUMMARIZATION_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                    },
                ]
            )
        ]
    )
    print(msg.content)
    return msg.content


def generate_img_summaries(path: str) -> tuple[list[str], list[str]]:
    """
    Generate summaries and base64 encoded strings for images
    path: Path to list of .jpg files extracted by Unstructured
    """

    img_base64_list = []
    image_summaries = []
    image_path = []

    for root, _, files in os.walk(path):
        for img_file in sorted(files):
            if img_file.endswith(".png"):
                base64_image = resize_base64_image(
                    encode_image(os.path.join(root, img_file)), 0.7
                )
                img_base64_list.append(base64_image)
                full_path = os.path.join(root, img_file)
                relative_path = full_path.replace("/content/", "", 1)
                image_path.append(relative_path)
                image_summaries.append(image_summarize(base64_image))
                time.sleep(1)
    return img_base64_list, image_summaries, image_path

In [ ]:
IMAGES_PATH = "/content/XXXXXXXX/XXXXXXXX"  # @param {type:"string"}

In [ ]:
img_base64_list, image_summaries, image_path = generate_img_summaries(IMAGES_PATH)

In [ ]:
# @title Save images summaries
def save_image_summaries_to_jsonl(
    summaries: list[str], image_paths: list[str], output_path: str
):
    with open(output_path, "w", encoding="utf-8") as f:
        for summary, path in zip(summaries, image_paths):
            f.write(
                json.dumps({"summary": summary, "image_path": path}, ensure_ascii=False)
                + "\n"
            )

In [ ]:
save_image_summaries_to_jsonl(image_summaries, image_path, f"{FILE_NAME}.jsonl")

#### Summaries loading

In [ ]:
def load_image_summaries_and_resize(
    input_path: str, resize_ratio: float = 0.7
) -> tuple[list[str], list[str], list[str]]:
    summaries = []
    image_paths = []
    base64_images = []

    for line in open(input_path, "r", encoding="utf-8"):
        item = json.loads(line)
        relative_path = item["image_path"]
        full_path = os.path.join("/content", relative_path)

        base64_image = resize_base64_image(encode_image(full_path), resize_ratio)

        summaries.append(item["summary"])
        image_paths.append(relative_path)
        base64_images.append(base64_image)

    return summaries, image_paths, base64_images

In [ ]:
image_path, image_summaries, img_base64_list = load_image_summaries_and_resize(
    "/content/XXXXXX/XXXXXX.jsonl"
)

## Create Retriever

In [ ]:
INDEX_ID = "XXXXXXXXXXXXXXX"  # @param {type: "string"}
ENDPOINT_ID = "XXXXXXXXXXXXXXX"  # @param {type: "string"}

In [ ]:
# The vectorstore to use to index the summaries
vectorstore = VectorSearchVectorStore.from_components(
    project_id=PROJECT_ID,
    region=LOCATION,
    gcs_bucket_name=GCS_BUCKET,
    index_id=INDEX_ID,
    endpoint_id=ENDPOINT_ID,
    embedding=VertexAIEmbeddings(model_name=EMBEDDING_MODEL_NAME),
    stream_update=True,
)

In [ ]:
try:
    storage_client = storage.Client()
    bucket = storage_client.bucket(GCS_BUCKET)
    if not bucket.exists():
        raise ValueError(f"Bucket '{GCS_BUCKET}' does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
docstore = GCSDocumentStorage(bucket, "chunks")

id_key = "doc_id"
retriever_multi_vector_img = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    id_key=id_key,
)

## Load Documents to Indexes

In [ ]:
def sanitize_metadata(metadata):
    sanitized_metadata = {}
    for key, value in metadata.items():
        if isinstance(value, bytes):
            sanitized_metadata[key] = value.decode("utf-8")
        else:
            sanitized_metadata[key] = value
    return sanitized_metadata

In [ ]:
chunks_documents = chunks_with_updated_metadata + [
    Document(
        page_content=img_base64,
        metadata={
            id_key: str(uuid.uuid4()),
            "url": urllib.parse.quote(os.path.join(GCS_BUCKET, image_path[i])),
        },
    )
    for i, img_base64 in enumerate(img_base64_list)
]

doc_ids = [doc.metadata[id_key] for doc in chunks_documents]

retriever_multi_vector_img.docstore.mset(list(zip(doc_ids, chunks_documents)))

In [ ]:
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(image_summaries)
] + [
    Document(page_content=doc.page_content, metadata={"doc_id": doc.metadata["doc_id"]})
    for doc in chunks_with_updated_metadata
]


def batch(iterable, batch_size=1000):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i : i + batch_size]


# Split to batches with a max size of 1,000
batch_size = 1000

batches = list(batch(summary_docs, batch_size))

for batch_docs in batches:
    document_ids = retriever_multi_vector_img.vectorstore.add_documents(batch_docs)

### Save chunk_id & document_id in a bucket inside the specific file folder

In [ ]:
def split_gcs_path(gs_path):
    gs_path = gs_path.removeprefix("gs://")
    return gs_path.split("/", 1)


def get_bucket(gs_path):
    try:
        return storage.Client().bucket(split_gcs_path(gs_path)[0])
    except Exception as error:
        raise Exception(error)


def get_blob(gs_path):
    try:
        return get_bucket(gs_path).blob(split_gcs_path(gs_path)[1])
    except Exception as error:
        raise Exception(error)


def write_to_json_file(data, folder_name):
    try:
        blob = get_blob(folder_name)
        json_data = json.dumps(data, indent=2, ensure_ascii=False)
        print(json_data)
        blob.upload_from_string(json_data, content_type="application/json")
    except Exception as error:
        raise Exception(error)

In [ ]:
def save_chunk_ids(chunks_for_index):
    chunk_ids = [doc.metadata["doc_id"] for doc in chunks_for_index]
    write_to_json_file(chunk_ids, CHUNK_IDS_PATH)

In [ ]:
save_chunk_ids(summary_docs)
write_to_json_file(document_ids, DOCUMENT_IDS_PATH)

## Delete Specific Chunks

In [ ]:
def load_doc_ids(gs_path):
    try:
        blob = get_blob(gs_path)
        json_data = blob.download_as_text(encoding="utf-8")
        return json.loads(json_data)
    except Exception as error:
        raise Exception(error)

### Delete Chunks

In [ ]:
def delete_chunks_from_docstore(docstore):
    chunk_ids = load_doc_ids(CHUNK_IDS_PATH)
    try:
        docstore.mdelete(chunk_ids)
        print(f"✅ Deleted {len(chunk_ids)} chunks from docstore.")
    except Exception as error:
        print(f"Failed to delete chunk_ids from docstore: {error}")

In [ ]:
delete_chunks_from_docstore(retriever_multi_vector_img.docstore)

### Delete Documents

In [ ]:
def delete_doc_ids_from_vectorstore(vectorstore):
    doc_ids = load_doc_ids(DOCUMENT_IDS_PATH)
    key_deleted_from_vectorstore = []

    for doc_id in doc_ids:
        try:
            vectorstore.delete([doc_id])
            key_deleted_from_vectorstore.append(doc_id)

        except Exception as error:
            print(f"Failed to delete {doc_id} from vectorstore: {error}")
            continue
    return key_deleted_from_vectorstore

In [ ]:
key_deleted_from_vectorstore = delete_doc_ids_from_vectorstore(
    retriever_multi_vector_img.vectorstore
)